# Data & Biases


Every recommender is a function of its training data, and recommender training data is **not** what an unbiased sample of user opinion would look like. A rating of 5 stars is not just a noisy measure of *how good the movie is* — it is also a measure of *who chose to watch that movie*, *what surfaced it to them*, and *what competition it faced*. The same row encodes the user's preference, the system's past behaviour, and the social proof around the title. If you forget that, every metric you compute over held-out ratings will be optimistic, every A/B test will be confounded, and every off-policy estimator will be biased.

This notebook grounds the recsys stack in data: how to load MovieLens, how to split it for offline evaluation, and how to quantify the biases that downstream notebooks will have to live with. The implementation lives in `notebooks.recsys.data`; every later notebook imports the `Dataset`, `time_split`, `popularity_bias_stats` we build here.


## Setup

Imports and plotting defaults. We seed everything so the figures are reproducible.


In [ ]:
#| echo: false
import warnings
warnings.filterwarnings("ignore")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline

backend_inline.set_matplotlib_formats("svg")
plt.rcParams["figure.dpi"] = 110

from notebooks.recsys.config import MovieLensConfig
from notebooks.recsys.data import load_movielens, time_split, popularity_bias_stats, sparsity


## Explicit vs implicit feedback

MovieLens is the canonical **explicit-feedback** dataset: the user rates a movie on a 1-5 scale. Formally, each row is a triple $(u, i, r_{ui}) \in \mathbb{Z} \times \mathbb{Z} \times \{1, \frac{1}{2}, \ldots, 5\}$ with an attached timestamp $t_{ui}$. The recommender's job is to predict $\hat{r}_{ui}$ and rank items by that prediction.

Most real systems — clicks, watches, listens — produce **implicit feedback**: a binary signal $c_{ui} \in \{0, 1\}$ of *did the user interact at all*, with no graded notion of how much they liked it. Implicit data is vastly more abundant and removes the *self-selection bias* of "only motivated users rate", but loses the rating magnitude.

Hu, Koren & Volinsky (2008) treat implicit feedback as a confidence-weighted observation:

$$\hat{r}_{ui} = \mathbf{p}_u^{\top} \mathbf{q}_i, \qquad c_{ui} = 1 + \alpha \cdot r_{ui}, \qquad p_{ui} = \begin{cases} 1 & r_{ui} > 0 \\ 0 & \text{otherwise} \end{cases}$$

and train weighted matrix factorization $\mathcal{L} = \sum_{u,i} c_{ui}(p_{ui} - \hat{r}_{ui})^2$. The confidence term $c_{ui}$ encodes "how many times the user clicked" — multiple listens are stronger evidence of preference than one. We will work with explicit ratings here in REC:01 (so EDA is interpretable) and switch to implicit interpretation in REC:03 (ALS) and REC:04 (the two-tower softmax).


:::{.callout-note}
The dichotomy is false. In production, the same log row carries both signals: clicks are implicit but dwell time and skip-after-N-seconds tell you whether the click was *positive*. Real systems treat this as an ordinal outcome and train a multi-class ranker (REC:05).
:::


## Loading MovieLens

`load_movielens` returns a `Dataset` dataclass with the ratings frame, movie metadata, and the id-to-index maps needed by every model later in the course. We default to `ml-100k` because the dataset is small (5 MB), can be fetched in under a second, and is enough to illustrate every concept in stages 1–9. Switch to `ml-25m` for any model that needs a few million interactions.


In [ ]:
cfg = MovieLensConfig(name="ml-100k")  # try "ml-25m" for the full snapshot
ds = load_movielens(cfg)
print(ds.summary())
ds.ratings.head()


**What landed in `ds.ratings`.** A four-column frame plus the integer `user_idx` / `item_idx` columns that every later stage uses to look up token IDs. The `user_index` / `item_index` maps are kept on the dataset object — never instantiate a model without them, or you lose the id ↔ token correspondence that makes train and serve use the same vocabulary.


In [ ]:
print(f"users: {ds.n_users:6d}")
print(f"items: {ds.n_items:6d}")
print(f"ratings: {ds.n_ratings:,}")
print(f"|matrix| = {ds.n_users * ds.n_items:,} cells")
print(f"sparsity = {sparsity(ds.ratings):.4%}")
print()
print("rating distribution:")
print(ds.ratings["rating"].value_counts().sort_index())


**Observations.** Even with $\sim 10^5$ ratings from MovieLens 100k the matrix is $\approx 93\%$ empty. That number grows to $99.99\%$ for the 25M snapshot. Sparsity is why matrix factorization, embedding models, and hierarchical priors work at all — the dense-matrix view of the same problem is computationally and statistically intractable.

MovieLens ratings are also concentrated at the high end: very few 1-star ratings exist because users self-curate before rating. The bias shows up immediately as the mean of the rating distribution shifting toward $3.5-4.0$. A ranker that predicts the global mean looks like a strong baseline under RMSE, and like a useless recommender under Recall@K.


## Time-based split

A train/test split random by row is **wrong** for recsys. Here is why.

Let the dataset be $\mathcal{D} = \{(u_i, m_i, r_i, t_i)\}_{i=1}^N$. A random row split partitions $\mathcal{D}$ into $\mathcal{D}_{train}$ and $\mathcal{D}_{val}$ by Bernoulli sampling. For a fixed user $u$, this means $\mathcal{D}_{train}^{(u)}$ contains ratings from both before and after every rating in $\mathcal{D}_{val}^{(u)}$. The model has seen the future.

Concretely: imagine user $u$ rated Toy Story in 2010, Inception in 2010, and The Batman in 2020. A random split puts The Batman in training and Toy Story in validation. The model evaluates "predict Toy Story" with full knowledge of everything the user rated up to 2020 — including every movie they watched *because* they liked Toy Story. Recall@K becomes a measure of *interpolation*, not *extrapolation*. Off-policy eval and A/B tests are the only honest accountings of the latter, and both require time-ordered splits as their starting point.

The correct split is per-user time-based: take each user's most recent `val_frac` of ratings as validation, the older 1 - `val_frac` as training. The implementation in `time_split` enforces this:


In [ ]:
train, val = time_split(ds.ratings, val_frac=0.2)
print(f"train: {len(train):,} rows")
print(f"val:   {len(val):,} rows")

# sanity: every train timestamp per user must precede every val timestamp
ok = train.groupby("user_id")["timestamp"].max().le(
    val.groupby("user_id")["timestamp"].min()
).all()
print(f"\nno future leakage?  {ok}")


Users with fewer than $\lceil 1 / \mathrm{val\_frac} \rceil$ ratings (i.e. fewer than 5 for `val_frac=0.2`) cannot be split meaningfully; they fall entirely into training. This avoids the trap of validating a model on users from whom it could not have learned anything.


## The three biases

Recommender training data is biased in three distinct, well-understood ways. They get conflated in colloquial use; they should not be.

**Selection bias.** A user only rates items they actually watched. Whether they watched an item depends on past recommendations, algorithmic surface, and social signals. The observed $r_{ui}$ is therefore a sample from $p(r_{ui} \mid u \text{ was shown } i, u \text{ chose to engage})$, not from $p(r_{ui})$. Marlin et al. call this the *Missing-Not-At-Random* (MNAR) regime. Models that ignore it (most of them) are biased toward items that the past system already surfaced.

**Position bias.** Users click items near the top of the list at higher rates regardless of relevance. The observed click $c_{ui}$ is a sample from $p(c_{ui} \mid \mathrm{rank}(i \mid u)) \cdot p(\mathrm{click on relevant item})$. Decoupling the two factors is the entire premise of counterfactual evaluation with inverse propensity scoring (REC:08).

**Popularity bias.** The few popular items appear in nearly every user's history. Models that minimize log-loss over the dataset will disproportionately recommend them, amplifying the rich-get-richer dynamics of the catalog. Sheetra & Mehrotra (2019) call this the *closed feedback loop*. The two metrics you'll meet in REC:03 — **Coverage** and **Novelty** — exist specifically to measure whether a model has escaped this loop.


In [ ]:
stats_full = popularity_bias_stats(ds.ratings)
stats_train = popularity_bias_stats(train)
stats_val = popularity_bias_stats(val)

print(f"{'set':12s}  {'Gini':>7s}  {'top-10% share':>14s}")
for name, s in [("full dataset", stats_full), ("train", stats_train), ("validation", stats_val)]:
    print(f"{name:12s}  {s['gini']:>7.4f}  {s['top10_share']:>14.4f}")


**Interpretation.** The Gini coefficient here is the area between the Lorenz curve of per-item rating counts and the line of perfect equality. MovieLens 100k has $\approx 0.68$ — comparable to US household wealth. The top decile of items absorb roughly 40-50% of all ratings. As later stages add more expressive models, watch this number go *up* under naive training (popularity amplification) and *down* when we add novelty regularization, inverse-propensity weighting, or an explore/exploit policy in REC:08.


## Caveats and link forward

Two things this stage deliberately omits:

1. **Positive/unlabelled (PU) framing.** Implicit data has only positives (clicked) and a sea of unobserved (not clicked). Plenty of unobserved items are also *would-be positives*. The literature treats this either by negative sampling (REC:04) or by modelling $p(r_{ui} > 0)$ directly (BPR, REC:05). We will pick up both in their respective stages.
2. **Drift over time.** The same MovieLens snapshot spans 1997-1998 for ml-100k, twenty years for ml-25m. Most of that drift is in the item distribution. Time-based split keeps the *user* stationary but the catalog drifts. REC:07 will show how the feature store survives this; REC:08 how the bandit policy adapts.

What `Dataset` gives us is enough to start: a table of user-item interactions, an id-to-index map, a no-leakage train/val split, and three numbers quantifying the biases the rest of the course will fight. The next notebook turns that data into features that downstream models can actually consume.
